"""
# STEP 1 — SCRIPT OVERVIEW (Markdown-style description)

This script processes EPC energy simulation outputs for multiple buildings and converts them into energy cost results.

For each building folder:
- Reads `energy_results_tot.csv`
- Reads a global energy price lookup table (`fixed_energy_prices.csv`)
- Multiplies each energy value by the corresponding tariff (matched by fuel type and time step)
- Maps `electricity_exp_9` to `fixed_export` (all other mappings remain unchanged from the carbon script)
- Writes the results into `fixed_cost_results.csv` inside each building’s TOTAL folder

Key assumptions:
- Time steps align between energy and price lookup tables
- Column naming follows the provided mapping rules
- Each building folder is named using BUILDING_REFERENCE_NUMBER
"""

In [1]:
# STEP 2 — IMPORTS AND FILE PATHS
# Description: Load required libraries and define input/output paths

import os
import pandas as pd
from pathlib import Path

# Root directories
EPC_FILE = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_upgrade.csv"
BASELINE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Energy price lookup file (NEW)
COST_LOOKUP_FILE = "/Users/jakegehrung/Desktop/data_raw/tariff/fixed_energy_prices.csv"

In [2]:
# STEP 3 — LOAD ENERGY COST LOOKUP TABLE
# Description: Load and prepare the tariff lookup table for time-aligned cost conversion

cost_df = pd.read_csv(COST_LOOKUP_FILE)

# Ensure Time is usable for alignment
cost_df["Time"] = cost_df["Time"].astype(str)

# Set Time as index for fast lookup
cost_df = cost_df.set_index("Time")

# Preview check (optional)
cost_df.head()

,electricity,lpg,mineral_wood,mains_gas,oil,wood_logs,wood_pellets,wood_chips,coal,electricity_export
Time,,,,,,,,,,
00:30:00,0.000257,0.0001,0.000048,0.000036,0.000049,0.000051,0.000063,0.000037,0.000056,0.000091
01:00:00,0.000257,0.0001,0.000048,0.000036,0.000049,0.000051,0.000063,0.000037,0.000056,0.000091
01:30:00,0.000257,0.0001,0.000048,0.000036,0.000049,0.000051,0.000063,0.000037,0.000056,0.000091
02:00:00,0.000257,0.0001,0.000048,0.000036,0.000049,0.000051,0.000063,0.000037,0.000056,0.000091
02:30:00,0.000257,0.0001,0.000048,0.000036,0.000049,0.000051,0.000063,0.000037,0.000056,0.000091


In [3]:
# STEP 4 — DEFINE COLUMN MAPPING AND PROCESSING FUNCTION
# Description: Map energy columns to tariff columns and compute cost per building

# Map EPC energy columns -> cost tariff columns
COLUMN_MAP = {
    "electricity_exp_1": "electricity_export",
    "electricity_tot_1": "electricity",
    "lpg_tot_1": "lpg",
    "mineral_wood_tot_1": "mineral_wood",
    "mains_gas_tot_1": "mains_gas",
    "oil_tot_1": "oil",
    "wood_logs_tot_1": "wood_logs",
    "wood_pellets_tot_1": "wood_pellets",
    "wood_chips_tot_1": "wood_chips",
    "coal_tot_1": "coal",

    "electricity_exp_2": "electricity_export",
    "electricity_tot_2": "electricity",
    "lpg_tot_2": "lpg",
    "mineral_wood_tot_2": "mineral_wood",
    "mains_gas_tot_2": "mains_gas",
    "oil_tot_2": "oil",
    "wood_logs_tot_2": "wood_logs",
    "wood_pellets_tot_2": "wood_pellets",
    "wood_chips_tot_2": "wood_chips",
    "coal_tot_2": "coal",

    "electricity_exp_3": "electricity_export",
    "electricity_tot_3": "electricity",
    "lpg_tot_3": "lpg",
    "mineral_wood_tot_3": "mineral_wood",
    "mains_gas_tot_3": "mains_gas",
    "oil_tot_3": "oil",
    "wood_logs_tot_3": "wood_logs",
    "wood_pellets_tot_3": "wood_pellets",
    "wood_chips_tot_3": "wood_chips",
    "coal_tot_3": "coal",

    "electricity_exp_4": "electricity_export",
    "electricity_tot_4": "electricity",
    "lpg_tot_4": "lpg",
    "mineral_wood_tot_4": "mineral_wood",
    "mains_gas_tot_4": "mains_gas",
    "oil_tot_4": "oil",
    "wood_logs_tot_4": "wood_logs",
    "wood_pellets_tot_4": "wood_pellets",
    "wood_chips_tot_4": "wood_chips",
    "coal_tot_4": "coal",

    "electricity_exp_5": "electricity_export",
    "electricity_tot_5": "electricity",
    "lpg_tot_5": "lpg",
    "mineral_wood_tot_5": "mineral_wood",
    "mains_gas_tot_5": "mains_gas",
    "oil_tot_5": "oil",
    "wood_logs_tot_5": "wood_logs",
    "wood_pellets_tot_5": "wood_pellets",
    "wood_chips_tot_5": "wood_chips",
    "coal_tot_5": "coal",

    "electricity_exp_6": "electricity_export",
    "electricity_tot_6": "electricity",
    "lpg_tot_6": "lpg",
    "mineral_wood_tot_6": "mineral_wood",
    "mains_gas_tot_6": "mains_gas",
    "oil_tot_6": "oil",
    "wood_logs_tot_6": "wood_logs",
    "wood_pellets_tot_6": "wood_pellets",
    "wood_chips_tot_6": "wood_chips",
    "coal_tot_6": "coal",

    "electricity_exp_7": "electricity_export",
    "electricity_tot_7": "electricity",
    "lpg_tot_7": "lpg",
    "mineral_wood_tot_7": "mineral_wood",
    "mains_gas_tot_7": "mains_gas",
    "oil_tot_7": "oil",
    "wood_logs_tot_7": "wood_logs",
    "wood_pellets_tot_7": "wood_pellets",
    "wood_chips_tot_7": "wood_chips",
    "coal_tot_7": "coal",

    "electricity_exp_8": "electricity_export",
    "electricity_tot_8": "electricity",
    "lpg_tot_8": "lpg",
    "mineral_wood_tot_8": "mineral_wood",
    "mains_gas_tot_8": "mains_gas",
    "oil_tot_8": "oil",
    "wood_logs_tot_8": "wood_logs",
    "wood_pellets_tot_8": "wood_pellets",
    "wood_chips_tot_8": "wood_chips",
    "coal_tot_8": "coal",

    "electricity_exp_9": "electricity_export",
    "electricity_tot_9": "electricity",
    "lpg_tot_9": "lpg",
    "mineral_wood_tot_9": "mineral_wood",
    "mains_gas_tot_9": "mains_gas",
    "oil_tot_9": "oil",
    "wood_logs_tot_9": "wood_logs",
    "wood_pellets_tot_9": "wood_pellets",
    "wood_chips_tot_9": "wood_chips",
    "coal_tot_9": "coal"    
}


def process_building_cost(building_id: str):
    """
    Process a single building folder and generate fixed_cost_results.csv
    """

    total_path = os.path.join(BASELINE_DIR, building_id, "TOTAL")

    energy_file = os.path.join(total_path, "energy_results_tot.csv")
    output_file = os.path.join(total_path, "fixed_cost_results.csv")

    if not os.path.exists(energy_file):
        print(f"[SKIP] Missing: {energy_file}")
        return

    energy_df = pd.read_csv(energy_file)
    energy_df["Time"] = energy_df["Time"].astype(str)
    energy_df = energy_df.set_index("Time")

    # Prepare output dataframe
    cost_out = pd.DataFrame(index=energy_df.index)

    # Iterate through energy columns
    for col in energy_df.columns:
        if col not in COLUMN_MAP:
            continue

        fuel = COLUMN_MAP[col]

        if fuel not in cost_df.columns:
            print(f"[WARNING] Missing cost factor for fuel: {fuel}")
            continue

        # Align by Time index
        price_factor = cost_df[fuel].reindex(energy_df.index)

        cost_out[col] = energy_df[col] * price_factor

    # Save result
    cost_out.reset_index().to_csv(output_file, index=False)
    print(f"[DONE] {building_id}")

In [4]:
# STEP 5 — MANUAL TEST (OPTIONAL)
# Description: Run a single building to validate cost conversion logic

test_building_id = "1001829067"

print("Testing with building:", test_building_id)

process_building_cost(test_building_id)

Testing with building: 1001829067
[DONE] 1001829067


In [5]:
# STEP 6 — RUN FOR ALL BUILDINGS
# Description: Batch process all EPC buildings into fixed energy cost outputs

epc_df = pd.read_csv(EPC_FILE)

building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].dropna().astype(str).unique()

print(f"Found {len(building_ids)} buildings")

for b_id in building_ids:
    process_building_cost(b_id)

print("All processing complete.")

Found 117 buildings
[DONE] 1001829067
[DONE] 1001951858
[DONE] 1001829071
[DONE] 1234761001
[DONE] 1001991633
[DONE] 1001664929
[DONE] 1001829059
[DONE] 1001063639
[DONE] 1234761000
[DONE] 1236594950
[DONE] 1001664925
[DONE] 1001906271
[DONE] 1000238911
[DONE] 1001829057
[DONE] 1234760999
[DONE] 1002143357
[DONE] 1001951854
[DONE] 1001829069
[DONE] 1002313096
[DONE] 1002143351
[DONE] 1001870854
[DONE] 1001870864
[DONE] 1002143293
[DONE] 1002143352
[DONE] 1002143353
[DONE] 1234647955
[DONE] 1002313093
[DONE] 1001991629
[DONE] 1001991628
[DONE] 1000238920
[DONE] 1001829085
[DONE] 1001063646
[DONE] 1001829058
[DONE] 1234806523
[DONE] 1001664941
[DONE] 1236034494
[DONE] 1000336709
[DONE] 1234761002
[DONE] 1002143355
[DONE] 1001906269
[DONE] 1001870855
[DONE] 1001664944
[DONE] 1000336711
[DONE] 1001829079
[DONE] 1001870859
[DONE] 1001664928
[DONE] 1234806526
[DONE] 1001951889
[DONE] 1001627558
[DONE] 1235812262
[DONE] 1001627567
[DONE] 1001627542
[DONE] 1001627549
[DONE] 1002143348
[DONE] 1